<a href="https://colab.research.google.com/github/Selvapriya05/Selvapriya-Codeboosters-2026/blob/main/Day_4/Day_4_Bigdata_PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Cell-1 Install PySpark

!pip install pyspark --quiet
print('PySpark installation complete')


PySpark installation complete


In [ ]:
#Step-2 - Import PySpark modules and create SparkSession

from pyspark.sql import SparkSession
from pyspark.sql import functions as functions
from pyspark.sql.functions import year,month, to_date, col, round as spark_round
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

#Create SparkSession

spark = SparkSession.builder \
      .appName('Day4_BigData_Sales') \
      .config('spark.sql.adaptive.enabled', 'true') \
      .getOrCreate()

print(f'Spark version  : {spark.version}')
print(f'SparkSession   : ACTIVE')
print(f'Application    : {spark.sparkContext.appName}')



Spark version  : 4.0.2
SparkSession   : ACTIVE
Application    : Day4_BigData_Sales


In [ ]:
#Step-3 - Load csv into PySpark Dataframe(BRONZE)

df_bronze = spark.read \
     .option('header', 'true') \
     .option('inferSchema', 'true') \
     .csv('large_sales_data.csv')

print('=== BRONZE LAYER -- Raw Data ===')
print(f'Rows    : {df_bronze.count()}')
print(f'Columns : {len(df_bronze.columns)}')
print(f'Names   : {df_bronze.columns}')
print()
df_bronze.printSchema()



=== BRONZE LAYER -- Raw Data ===
Rows    : 5000
Columns : 13
Names   : ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'revenue', 'order_date', 'city', 'region', 'sales_rep', 'payment_method', 'order_status']

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [ ]:
#Cell-4 - Inspect first rows and summary statistics

print('First 5 rows:')
df_bronze.show(5, truncate=False)

print('\nBasic statistics for numeric columns:')
df_bronze.select('quantity', 'unit_price', 'revenue').describe().show()
#df_bronze('unit_price' + 'revenue')

First 5 rows:
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi Ku

In [ ]:
from posixpath import dirname
#CELL5- Save Bronze layer as  Parquet

df_bronze.write \
    .mode('overwrite') \
    .parquet('sales_bronze.parquet')

print('Bronze Parquet saved : sales_bronze.parquet')

#compare file sizes
import os

def get_dir_size(path):
   """Get total size of a file or directory in KB."""
   if os.path.isfile(path):
      return os.path.getsize(path) / 1024
   total = 0
   for dirpath, dirnames, filenames in os.walk(path):
       for f in filenames:
           total += os.path.getsize(os.path.join(dirpath, f))
   return total / 1024

csv_size     = get_dir_size('large_sales_data.csv')
parquet_size = get_dir_size('sales_bronze.parquet')
reduction    = (1 - parquet_size / csv_size) * 100

print(f'\nCSV file size   : {csv_size:.1f} KB')
print(f'Parquet file size : {parquet_size:.1f} KB')
print(f'Reduction         : {reduction:.1f}% smaller')
print(f'\nAt 1 TB scale: CSV=1000 GB -> Parquet={1000*(1-reduction/100):.0f} GB')

Bronze Parquet saved : sales_bronze.parquet

CSV file size   : 529.3 KB
Parquet file size : 55.1 KB
Reduction         : 89.6% smaller

At 1 TB scale: CSV=1000 GB -> Parquet=104 GB


In [ ]:
#STEP-6 - SILVER: Clean and enrich the data

from pyspark.sql import functions as F
#from pyspark.sql.functions import col, to_date, year, month

df_silver = df_bronze \
    .dropDuplicates() \
    .dropna(subset=['order_id','product','revenue'])

df_silver = df_silver.withColumn(
    'order_date',
    to_date(col('order_date'), 'dd-MM-yyyy')
)

df_silver = df_silver \
    .withColumn('order_year', year(col('order_date'))) \
    .withColumn('order_month', month(col('order_date')))

df_silver = df_silver.withColumn(
    'revenue_category',
    F.when(col('revenue') > 40000, 'High')
    .when(col('revenue') > 20000, 'Medium')
    .otherwise('Low')
)

print(f'Silver layer rows    : {df_silver.count()}')
print('New columns added: order_year, order_month, revenue_category')
df_silver.select('order_date', 'order_year', 'order_month', 'revenue_category').show(8)


Silver layer rows    : 5000
New columns added: order_year, order_month, revenue_category
+----------+----------+-----------+----------------+
|order_date|order_year|order_month|revenue_category|
+----------+----------+-----------+----------------+
|2023-02-07|      2023|          2|             Low|
|2023-01-24|      2023|          1|             Low|
|2023-04-16|      2023|          4|            High|
|2023-12-21|      2023|         12|             Low|
|2023-08-23|      2023|          8|            High|
|2023-05-26|      2023|          5|          Medium|
|2023-11-04|      2023|         11|          Medium|
|2023-01-24|      2023|          1|            High|
+----------+----------+-----------+----------------+
only showing top 8 rows


In [ ]:
#Step7 - Save Silver layer as Parquet

df_silver.write \
    .mode('overwrite') \
    .parquet('sales_silver.parquet')

print('Silver Parquet saved : sales_silver.parquet')
print(f'Silver size: {get_dir_size("sales_silver.parquet"):.1f} KB')
print(f'Bronze size: {get_dir_size("sales_bronze.parquet"):.1f} KB')

df_verify = spark.read.parquet('sales_silver.parquet')

print(f'Read-back rows: {df_verify.count()} (should match Silver count)')
df_verify.printSchema()


Silver Parquet saved : sales_silver.parquet
Silver size: 59.8 KB
Bronze size: 55.1 KB
Read-back rows: 5000 (should match Silver count)
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_category: string (nullable = true)



In [ ]:
#Step8 - Query1 - Top 5 products by total revenue

top_products = df_silver \
    .groupBy('product') \
    .agg(
        F.sum('revenue').alias('total_revenue'),
        F.count('order_id').alias('num_orders'),
        F.avg('revenue').alias('avg_order_revenue')
    ) \
    .orderBy('total_revenue', ascending=False) \
    .limit(5)

print(' === Top 5 products by Total Revenue  === ')
top_products.show(truncate=False)



 === Top 5 products by Total Revenue  === 
+-------+-------------+----------+------------------+
|product|total_revenue|num_orders|avg_order_revenue |
+-------+-------------+----------+------------------+
|Laptop |182700000    |502       |363944.22310756973|
|Tablet |135104000    |532       |253954.8872180451 |
|Monitor|82126000     |481       |170740.12474012474|
|Printer|44544000     |488       |91278.68852459016 |
|Speaker|16317000     |470       |34717.02127659575 |
+-------+-------------+----------+------------------+



In [ ]:
#Step9 - Query2

monthly_summary = df_silver \
    .groupBy('order_month') \
    .agg(
        F.sum('revenue').alias('monthly_revenue'),
        F.count('order_id').alias('monthly_orders')
    ) \
    .orderBy('order_month')

monthly_summary.show(truncate=False)

+-----------+---------------+--------------+
|order_month|monthly_revenue|monthly_orders|
+-----------+---------------+--------------+
|1          |41068200       |423           |
|2          |34485400       |375           |
|3          |40031200       |451           |
|4          |38857100       |390           |
|5          |39984500       |423           |
|6          |40707400       |390           |
|7          |42640700       |405           |
|8          |43718500       |418           |
|9          |37640200       |398           |
|10         |47839000       |479           |
|11         |44577100       |419           |
|12         |44298300       |429           |
+-----------+---------------+--------------+



In [ ]:
#Step10 - Query3

monthly_summary = df_silver \
    .groupBy(F.date_format("order_date", "MMMM").alias("order_month")) \
    .agg(
        F.sum('revenue').alias('monthly_revenue'),
        F.count('order_id').alias('monthly_orders')
    ) \
    .orderBy(F.min("order_date"))

monthly_summary.show(truncate=False)

+-----------+---------------+--------------+
|order_month|monthly_revenue|monthly_orders|
+-----------+---------------+--------------+
|January    |41068200       |423           |
|February   |34485400       |375           |
|March      |40031200       |451           |
|April      |38857100       |390           |
|May        |39984500       |423           |
|June       |40707400       |390           |
|July       |42640700       |405           |
|August     |43718500       |418           |
|September  |37640200       |398           |
|October    |47839000       |479           |
|November   |44577100       |419           |
|December   |44298300       |429           |
+-----------+---------------+--------------+

